# Risk Management Gold Layer

This layer consolidates portfolio-level risk metrics per fund.

It represents the current "state of risk" for each fund, including:
- market risk (return, volatility, VaR)
- stress risk scenarios
- liquidity risk (rescue constraints vs market liquidity)
- portfolio structure metrics

This dataset will serve as the baseline for simulation and ML models.

---

## Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Read Data

In [0]:
def read_table(table_name):
    """
    Read a Delta table from the Spark catalog.

    Args:
        table_name (str): Name of the table to read.

    Returns:
        pyspark.sql.DataFrame: Spark DataFrame loaded from the table.
    """
    return spark.table(table_name)

In [0]:
def get_latest_market_state(market_df):
    """
    Keep only the latest observation per ticker.

    Args:
        market_df (pyspark.sql.DataFrame): Spark DataFrame with market data.

    Returns:
        pyspark.sql.DataFrame: DataFrame containing only the most recent
        record per ticker.
    """
    window_spec = Window.partitionBy("ticker").orderBy(F.col("date").desc())

    return (
        market_df
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )


In [0]:
def join_portfolio(funds_df, market_df):
    """
    Join fund holdings with market data at ticker level.

    Args:
        funds_df (pyspark.sql.DataFrame): Fund portfolio positions.
        market_df (pyspark.sql.DataFrame): Market data at ticker level.

    Returns:
        pyspark.sql.DataFrame: Joined DataFrame containing portfolio
        and market attributes.
    """
    return funds_df.join(market_df, on="ticker", how="inner")


In [0]:
def calculate_liquidity(df, market_df):
    """
    Estimate asset-level liquidity proxy based on price and volume.

    Args:
        df (pyspark.sql.DataFrame): Portfolio DataFrame with ticker-level holdings.
        market_df (pyspark.sql.DataFrame): Market data with liquidity inputs.

    Returns:
        pyspark.sql.DataFrame: DataFrame enriched with asset-level liquidity
        and weighted liquidity contribution.
    """
    market_df = market_df.withColumn(
        "liquidity_days_asset",
        F.col("close") * F.col("avg_volume_30d")
    )

    df = df.join(
        market_df.select("ticker", "liquidity_days_asset"),
        on="ticker",
        how="left"
    )

    return df.withColumn(
        "weighted_liquidity",
        F.col("weight_pct") * F.col("liquidity_days_asset")
    )

In [0]:
def aggregate_portfolio_metrics(df):
    """
    Aggregate asset-level metrics into fund-level risk indicators.

    Args:
        df (pyspark.sql.DataFrame): Asset-level enriched portfolio data.

    Returns:
        pyspark.sql.DataFrame: Fund-level aggregated risk metrics.
    """
    df = df.withColumn(
        "weighted_return",
        F.col("weight_pct") * F.col("return_mean_30d")
    ).withColumn(
        "weighted_volatility",
        F.col("weight_pct") * F.col("volatility_30d")
    )

    return df.groupBy("fund_id", "fund_pl", "redemption_days").agg(
        F.sum("weight_pct").alias("check_weight_sum"),
        F.sum("weighted_return").alias("portfolio_return_mean"),
        F.sum("weighted_volatility").alias("portfolio_volatility"),
        F.sum("position_value").alias("portfolio_exposure"),
        F.avg("avg_volume_30d").alias("portfolio_avg_volume"),
        F.avg("volume_std_30d").alias("portfolio_volume_risk"),
        F.avg("weighted_liquidity").alias("portfolio_liquidity_days")
    )

In [0]:
def calculate_var(df):
    """
    Compute Value-at-Risk (VaR) proxy using volatility scaling.

    Args:
        df (pyspark.sql.DataFrame): Fund-level risk metrics DataFrame.

    Returns:
        pyspark.sql.DataFrame: DataFrame enriched with VaR estimate.
    """
    return df.withColumn(
        "portfolio_var_95",
        F.col("portfolio_volatility") * 1.65
    )


In [0]:
def calculate_stress(df):
    """
    Apply deterministic stress scenarios to portfolio exposure.

    Args:
        df (pyspark.sql.DataFrame): Fund-level risk metrics DataFrame.

    Returns:
        pyspark.sql.DataFrame: DataFrame with stress loss scenarios added.
    """
    return (
        df
        .withColumn("stress_loss_10pct", F.col("portfolio_exposure") * 0.10)
        .withColumn("stress_loss_20pct", F.col("portfolio_exposure") * 0.20)
        .withColumn("stress_loss_30pct", F.col("portfolio_exposure") * 0.30)
    )

In [0]:
def calculate_liquidity_risk(df):
    """
    Compute liquidity risk by comparing liquidity horizon with redemption rules.

    Args:
        df (pyspark.sql.DataFrame): Fund-level risk metrics DataFrame.

    Returns:
        pyspark.sql.DataFrame: DataFrame with liquidity gap and risk flag.
    """
    return (
        df
        .withColumn(
            "liquidity_gap",
            F.col("portfolio_liquidity_days") - F.col("redemption_days")
        )
        .withColumn(
            "liquidity_risk_flag",
            F.col("liquidity_gap") < 0
        )
    )


In [0]:
def add_structure_metrics(df):
    """
    Add structural portfolio risk indicators.

    Args:
        df (pyspark.sql.DataFrame): Fund-level risk metrics DataFrame.

    Returns:
        pyspark.sql.DataFrame: DataFrame with additional structure-based risk score.
    """
    return df.withColumn(
        "risk_score_proxy",
        F.col("portfolio_volatility") * F.col("portfolio_exposure")
    )

In [0]:
def main():
    """
    Execute the full risk management pipeline from raw market data to
    fund-level risk metrics.

    Steps:
        1. Load funds and market datasets
        2. Select latest market observations
        3. Join portfolio with market data
        4. Compute liquidity proxies
        5. Aggregate to fund-level metrics
        6. Compute VaR, stress, liquidity risk and structure metrics

    Returns:
        None
    """
    funds = read_table("risk_management.funds")
    market = read_table("risk_management.risk_dataset_silver")

    market_latest = get_latest_market_state(market)

    df = join_portfolio(funds, market_latest)

    df = calculate_liquidity(df, market_latest)

    df = aggregate_portfolio_metrics(df)

    df = calculate_var(df)
    df = calculate_stress(df)

    df = calculate_liquidity_risk(df)
    df = add_structure_metrics(df)

    display(df)



In [0]:
if __name__ == "__main__":
    main()

In [0]:
%sql
SELECT * FROM risk_management.funds;
